In [50]:
!pip install pdfplumber

In [1]:
import os
import time
import pdfplumber
import pandas as pd
import shutil


folder = r"C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Daily Documents"
processed_folder_path = r"C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Process Documents"
not_processed_folder_path = r"C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Not Processed Documents"
master_excel_path = r"C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Master Excel\master.xlsx"




output_location = master_excel_path
print("Output location : ",output_location)

while True:

    # Get all PDF files
    pdf_files = [os.path.join(folder, file) for file in os.listdir(folder) if file.lower().endswith(".pdf")]
    
    if pdf_files:
        # Get the latest PDF
        latest_pdf = max(pdf_files, key=os.path.getmtime)
        print("Latest PDF:", latest_pdf)
    else:
        print("No PDF found.")

    time.sleep(10)      # Check every 10 seconds

    invoice_path = latest_pdf




    def final_df_and_full_text(invoice_path):
        # extracting all tables from the pdf
        all_tables = []
        
        with pdfplumber.open(invoice_path) as pdf:
            for page_no, page in enumerate(pdf.pages, start=1):
                tables = page.extract_tables({
                    "vertical_strategy": "lines",
                    "horizontal_strategy": "lines"
                })
        
                for tbl_id, table in enumerate(tables, start=1):
                    if table is None:
                        continue
        
                    clean_table = [
                        ["" if c is None else c for c in row]
                        for row in table
                    ]
        
                    df = pd.DataFrame(clean_table)
                    df.columns = [f"Col_{i+1}" for i in range(df.shape[1])]
        
                    df["page_no"] = page_no
                    df["table_id"] = tbl_id
        
                    all_tables.append(df)
        
        final_df = pd.concat(all_tables, ignore_index=True)
        
        # extracting all the line from the pdf
        with pdfplumber.open(invoice_path) as pdf:
            line_wise_line = ""
            for page in pdf.pages:
                line_wise_line += page.extract_text() + "\n"
        return final_df, line_wise_line
    
    table_data, line_wise_line = final_df_and_full_text(invoice_path)
    
    
    
    try :
        existing_df = pd.read_excel(master_excel_path, engine='openpyxl') 
        
        # Invoice No engine='openpyxl'
        invoice_number_line = line_wise_line.split("\n")[0]
        invoice_number = invoice_number_line.split(":")[1]
        print("Invoice Number : ",invoice_number)
        
        # Date of issue
        date_of_issue_line = line_wise_line.split("\n")[1]
        date_of_issue = date_of_issue_line.split(":")[1]
        print("Date of Issue : ",date_of_issue)
        
        # ["No.", "Description", "Qty", "UM", "Net Price", "Net Worth", "VAT %", "Gross Worth"] 
        
        columns_li = []
        
        for a in table_data.columns: 
            for b in range(2): 
                if "No." in str(table_data[a][b]):
                    columns_li.append(a) 
                elif "Description" in str(table_data[a][b]): 
                    columns_li.append(a)
                elif "Qty" in str(table_data[a][b]): 
                    columns_li.append(a)
                elif "UM" in str(table_data[a][b]):
                    columns_li.append(a)
                elif "Net Price" in str(table_data[a][b]):
                    columns_li.append(a)
                elif "Net Worth" in str(table_data[a][b]):
                    columns_li.append(a)
                elif "VAT %" in str(table_data[a][b]): 
                    columns_li.append(a)
                elif "Gross" in str(table_data[a][b]): 
                    columns_li.append(a)
        
        uncleaned_df_for_first_table = table_data[columns_li]
        # print("Total df : \n",uncleaned_df_for_first_table)
        
        for d in range(len(uncleaned_df_for_first_table)): 
            if "VAT %" in str(uncleaned_df_for_first_table["Col_2"][d]):
                row_index = d
        print("First df ends at row no : ",row_index)
        
        cleaned_df_of_items = uncleaned_df_for_first_table.iloc[:row_index,:]
        
        cleaned_df_of_items.columns = cleaned_df_of_items.iloc[0]
        cleaned_df_of_items = cleaned_df_of_items[1:].reset_index(drop=True)
        cleaned_df_of_items
        
        
        uncleaned_df_for_second_table = uncleaned_df_for_first_table.iloc[row_index:,:]
        # [ "VAT %", "Net Worth", "VAT", "Gross Worth"] 
        
        columns_li_for_2nd_table = []
        
        for a in uncleaned_df_for_second_table.columns: 
            for b in range(2): 
                if "VAT %" in str(uncleaned_df_for_second_table[a].iloc[b]):
                    columns_li_for_2nd_table.append(a) 
                elif "Net Worth" in str(uncleaned_df_for_second_table[a].iloc[b]): 
                    columns_li_for_2nd_table.append(a)
                elif "VAT" in str(uncleaned_df_for_second_table[a].iloc[b]): 
                    columns_li_for_2nd_table.append(a)
                elif "Gross Worth" in str(uncleaned_df_for_second_table[a].iloc[b]):
                    columns_li_for_2nd_table.append(a)
        
        cleaned_df_of_summary = uncleaned_df_for_second_table[columns_li_for_2nd_table]
        
        cleaned_df_of_summary.columns = cleaned_df_of_summary.iloc[0]
        cleaned_df_of_summary = cleaned_df_of_summary[1:].reset_index(drop=True)
        cleaned_df_of_summary = cleaned_df_of_summary.iloc[:1,:]
        # cleaned_df_of_summary
        
        
        columns_names = ["Invoice no",
                         "Date of issue",
                         "Item No",
                         "Item Description",
                         "Item Qty", 
                         "Item UM", 
                         "Item Net Worth",
                         "Item VAT %",
                         "Item Gross Worth", 
                         "Summary VAT %",
                         "Summary Net Worth", 
                         "Summary VAT", 
                         "Summary Gross Worth"]
        
        item_no_li = cleaned_df_of_items["No."].to_list()
        item_description_li = cleaned_df_of_items["Description"].to_list()
        item_qty_li = cleaned_df_of_items["Qty"].to_list()
        item_um_li = cleaned_df_of_items["UM"].to_list()
        item_net_price_li = cleaned_df_of_items["Net Price"].to_list()
        item_net_worth_li = cleaned_df_of_items["Net Worth"].to_list()
        item_vat_li = cleaned_df_of_items["VAT %"].to_list()
        item_gross_worth = cleaned_df_of_items["Gross\nWorth"].to_list()
        
        
        summary_vat_percent = cleaned_df_of_summary["VAT %"].iloc[0]
        summary_net_worth = cleaned_df_of_summary["Net Worth"].iloc[0] 
        summary_vat = cleaned_df_of_summary["VAT"].iloc[0] 
        summary_gross_worth = cleaned_df_of_summary["Gross Worth"].iloc[0]
        
        
        import pandas as pd
        df = pd.DataFrame(columns = columns_names)
        for no in range(len(item_no_li)): 
            df.loc[len(df),["Invoice no",
                         "Date of issue",
                         "Item No",
                         "Item Description",
                         "Item Qty", 
                         "Item UM", 
                         "Item Net Worth",
                         "Item VAT %",
                         "Item Gross Worth", 
                         "Summary VAT %",
                         "Summary Net Worth", 
                         "Summary VAT", 
                         "Summary Gross Worth"]] = [invoice_number, 
                                                    date_of_issue, 
                                                    item_no_li[no], 
                                                    item_description_li[no], 
                                                    item_qty_li[no], 
                                                    item_um_li[no], 
                                                    item_net_worth_li[no], 
                                                    item_vat_li[no], 
                                                    item_gross_worth[no], 
                                                    summary_vat_percent, 
                                                    summary_net_worth, 
                                                    summary_vat, 
                                                    summary_gross_worth]
        
        

            
        if not os.path.exists(master_excel_path): 
            df.to_excel(master_excel_path, index=False) 
        else: 
            existing_df = pd.read_excel( master_excel_path, engine='openpyxl' ) 
            final_df = pd.concat([existing_df, df], ignore_index=True) 
            final_df.to_excel(master_excel_path, index=False) 
            
            shutil.move(invoice_path, processed_folder_path) 
            print("PDF moved to processed folder.") 
 

        
    except Exception as e: 
        print(e)
        shutil.move(invoice_path, not_processed_folder_path)
        print("PDF moved to not processed folder.")
        

Output location :  C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Master Excel\master.xlsx
Latest PDF: C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Daily Documents\invoice_51109400.pdf
Invoice Number :   51109400
Date of Issue :   22/02/2024
First df ends at row no :  3
PDF moved to processed folder.
Latest PDF: C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Daily Documents\invoice_51109399.pdf
Invoice Number :   51109399
Date of Issue :   26/06/2023
First df ends at row no :  4
PDF moved to processed folder.
Latest PDF: C:\Users\yadav\Downloads\JSL Internship codes\JSL Internship Project\Daily Documents\invoice_51109398.pdf


KeyboardInterrupt: 

PDF moved successfully.
